In [ ]:
!nvidia-smi

In [1]:
!pip install ortools


[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 12.4 MB/s eta 0:00:00m eta 0:00:01:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 2.6 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for pycuda (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [4402 lines of output]
      ***************************************************************
      *** WARNING: nvcc not in path.
      *** May need to set CUDA_INC_DIR for installation to succeed.
      ***************************************************************
      *************************************************************
      *** I have detected that you have not run configure.py.
      ******************************************

In [3]:
from ortools.algorithms.python import knapsack_solver
import numpy as np
import time
import threading

In [ ]:
NUMBER_OF_TRIALS = 100
PREVIEW_ITEMS = 15

import numpy as np

import problems

# Every implementation in this project solves the identical set of problems by
# reading them from problems.py.
problem_set = problems.generate()

num_problems = problem_set.num_problems
num_items_per_problem = problem_set.num_items
max_capacity = problem_set.max_capacity
capacities = problem_set.capacities

# Subset sum: an item's value is its weight, so the solver cells below are
# handed the same array for both.
values = weights = problem_set.items

num_items = np.full(num_problems, num_items_per_problem, dtype=np.int32)
max_values = np.zeros(num_problems, dtype=np.int32)

print(f"{num_problems} problems, {num_items_per_problem} items each, capacity at most {max_capacity}")
for i in range(3):
    row = problem_set.items[i]
    head = ", ".join(str(item) for item in row[:PREVIEW_ITEMS])
    rest = f", ... ({len(row) - PREVIEW_ITEMS} more)" if len(row) > PREVIEW_ITEMS else ""
    print(f"Problem {i + 1}: capacity {problem_set.capacities[i]}, items [{head}{rest}]")

In [7]:
# OR-Tools Section

# This section should run in Colab T4
import platform
print(platform.node())

# Function to solve a single knapsack problem using OR-Tools
def solve_knapsack(values, weights, capacity, problem_idx, results):
    solver = knapsack_solver.KnapsackSolver(
        knapsack_solver.KNAPSACK_MULTIDIMENSION_BRANCH_AND_BOUND_SOLVER, 'KnapsackExample')

    solver.init(values, [weights], [capacity])

    max_value = solver.solve()
    results[problem_idx] = max_value
    #print(f"Problem {problem_idx + 1}: Maximum value = {max_value}")

# Storage for results
results = [0] * num_problems

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    # Create and start threads
    threads = []
    for i in range(num_problems):
        t = threading.Thread(target=solve_knapsack, args=(values[i], weights[i], capacities[i], i, results))
        threads.append(t)
        t.start()

    # Wait for all threads to complete
    for t in threads:
        t.join()

    # Print execution time
    end_time = time.time()
    #print(f"Threaded execution time: {end_time - start_time:.6f} seconds")
    execution_times.append(end_time - start_time)

print(f"Average CPU execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Print the final results
#for i in range(num_problems):
for i in range(10):
    print(f"Final Result for Problem {i + 1}: Maximum value = {results[i]}")


Andrews-Personal-MacBook.local
Average CPU execution time: 0.783799 seconds
Final Result for Problem 1: Maximum value = 145
Final Result for Problem 2: Maximum value = 61
Final Result for Problem 3: Maximum value = 162
Final Result for Problem 4: Maximum value = 88
Final Result for Problem 5: Maximum value = 53
Final Result for Problem 6: Maximum value = 0
Final Result for Problem 7: Maximum value = 38
Final Result for Problem 8: Maximum value = 81
Final Result for Problem 9: Maximum value = 156
Final Result for Problem 10: Maximum value = 49
